<a href="https://colab.research.google.com/github/DhimanTarafdar/dengue-virus-research/blob/main/Dengue_research_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install biopython mappy -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import pandas as pd
from Bio import SeqIO

df = pd.read_csv("/content/DENGU-VIRUS-CSV.csv")

seqs = {}
for rec in SeqIO.parse("/content/DENGUE-VIRUS-sequences.fasta", "fasta"):
    acc = rec.id.split(".")[0]
    seqs[acc] = str(rec.seq).upper()

print(df.shape)
print(len(seqs))

(664, 15)
664


In [3]:
print(set(df["Accession"]) == set(seqs))

df["seq_len"] = df["Accession"].map(lambda a: len(seqs[a]))
print((df["seq_len"] != df["Length"]).sum())

True
0


In [4]:
def get_serotype(name):
    last = name.lower().split()[-1]
    mapping = {"1": "DENV-1", "i": "DENV-1", "2": "DENV-2", "3": "DENV-3", "4": "DENV-4"}
    return mapping[last]

df["ncbi_serotype"] = df["Organism_Name"].apply(get_serotype)
print(df["ncbi_serotype"].value_counts())

ncbi_serotype
DENV-3    310
DENV-2    278
DENV-1     73
DENV-4      3
Name: count, dtype: int64


In [5]:
df["year"] = df["Collection_Date"].astype(str).str.extract(r"(\d{4})")[0]
df["year"] = pd.to_numeric(df["year"])

print(df["year"].isna().sum())
print(df["year"].value_counts().sort_index())

14
year
2001.0      1
2002.0      4
2004.0      1
2005.0      1
2006.0     12
2007.0      4
2008.0      7
2009.0      6
2011.0      1
2017.0    136
2018.0     90
2019.0     70
2020.0      1
2021.0    101
2022.0     78
2023.0     79
2024.0     20
2025.0     38
Name: count, dtype: int64


In [6]:
print(pd.crosstab(df["year"], df["ncbi_serotype"]))

ncbi_serotype  DENV-1  DENV-2  DENV-3  DENV-4
year                                         
2001.0              0       0       1       0
2002.0              0       0       4       0
2004.0              0       1       0       0
2005.0              0       1       0       0
2006.0              0       2      10       0
2007.0              0       1       3       0
2008.0              0       3       4       0
2009.0              1       3       2       0
2011.0              0       1       0       0
2017.0             14     105      17       0
2018.0              5      49      36       0
2019.0              4       2      64       0
2020.0              1       0       0       0
2021.0             43       0      58       0
2022.0              0       2      76       0
2023.0              0      74       5       0
2024.0              0      14       5       1
2025.0              5      20      11       2


In [7]:
complete = df[df["Nuc_Completeness"] == "complete"]

for sero in ["DENV-1", "DENV-2", "DENV-3", "DENV-4"]:
    with open(f"/content/ref_{sero}.fa", "w") as f:
        for acc in complete[complete["ncbi_serotype"] == sero]["Accession"]:
            f.write(f">{acc}\n{seqs[acc]}\n")

print(complete["ncbi_serotype"].value_counts())

ncbi_serotype
DENV-3    37
DENV-2    22
DENV-1     7
DENV-4     2
Name: count, dtype: int64


In [8]:
import mappy as mp

aligners = {}
for sero in ["DENV-1", "DENV-2", "DENV-3", "DENV-4"]:
    aligners[sero] = mp.Aligner(f"/content/ref_{sero}.fa", preset="sr", k=9, w=3)

def best_match(acc):
    seq = seqs[acc]
    scores = {}
    for sero, aligner in aligners.items():
        best = 0
        for hit in aligner.map(seq):
            if hit.ctg == acc:
                continue
            if hit.mlen > best:
                best = hit.mlen
        scores[sero] = best / len(seq)
    return scores

rows = []
for acc in df["Accession"]:
    s = best_match(acc)
    top = max(s, key=s.get)
    rows.append([acc, top, s[top]])

result = pd.DataFrame(rows, columns=["Accession", "checked_serotype", "match_score"])
df = df.merge(result, on="Accession")
print(df.shape)

(664, 20)


In [9]:
print(pd.crosstab(df["ncbi_serotype"], df["checked_serotype"]))

wrong = df[df["ncbi_serotype"] != df["checked_serotype"]]
print(len(wrong))
print(wrong["year"].value_counts())
print(wrong["Genotype"].value_counts(dropna=False))

checked_serotype  DENV-1  DENV-2  DENV-3  DENV-4
ncbi_serotype                                   
DENV-1                29       0      44       0
DENV-2                 0     278       0       0
DENV-3                 0       0     310       0
DENV-4                 0       0       0       3
44
year
2021.0    43
2020.0     1
Name: count, dtype: int64
Genotype
1    44
Name: count, dtype: int64


In [10]:
df["serotype"] = df["checked_serotype"]
df["label_wrong"] = df["ncbi_serotype"] != df["serotype"]

print(df["serotype"].value_counts())
print(df["label_wrong"].sum())

serotype
DENV-3    354
DENV-2    278
DENV-1     29
DENV-4      3
Name: count, dtype: int64
44


In [11]:
print(pd.crosstab(df["year"], df["serotype"]))

serotype  DENV-1  DENV-2  DENV-3  DENV-4
year                                    
2001.0         0       0       1       0
2002.0         0       0       4       0
2004.0         0       1       0       0
2005.0         0       1       0       0
2006.0         0       2      10       0
2007.0         0       1       3       0
2008.0         0       3       4       0
2009.0         1       3       2       0
2011.0         0       1       0       0
2017.0        14     105      17       0
2018.0         5      49      36       0
2019.0         4       2      64       0
2020.0         0       0       1       0
2021.0         0       0     101       0
2022.0         0       2      76       0
2023.0         0      74       5       0
2024.0         0      14       5       1
2025.0         5      20      11       2


In [12]:
audit = pd.crosstab(df["ncbi_serotype"], df["serotype"])
audit.loc["Total"] = audit.sum()
print(audit)

print(df[df["label_wrong"]][["Accession", "Organism_Name", "Length", "year", "Submitters"]].head(10))

serotype       DENV-1  DENV-2  DENV-3  DENV-4
ncbi_serotype                                
DENV-1             29       0      44       0
DENV-2              0     278       0       0
DENV-3              0       0     310       0
DENV-4              0       0       0       3
Total              29     278     354       3
    Accession        Organism_Name  Length    year  \
356  OQ520872  dengue virus type 1     395  2021.0   
357  OQ520873  dengue virus type 1     414  2021.0   
358  OQ520874  dengue virus type 1     414  2021.0   
359  OQ520875  dengue virus type 1     430  2021.0   
360  OQ520876  dengue virus type 1     430  2021.0   
361  OQ519654  dengue virus type 1    1479  2021.0   
362  OQ519655  dengue virus type 1    1479  2021.0   
363  OQ519656  dengue virus type 1    1479  2021.0   
364  OQ519657  dengue virus type 1    1479  2021.0   
365  OQ519658  dengue virus type 1    1479  2021.0   

                                        Submitters  
356  Jahan,A., Shyamal,K., Aun

In [13]:
df.to_csv("/content/dengue_step1_serotype.csv", index=False)

In [14]:
def get_position(acc, sero):
    seq = seqs[acc]
    best = None
    for hit in aligners[sero].map(seq):
        if hit.ctg == acc:
            continue
        if best is None or hit.mlen > best.mlen:
            best = hit
    if best is None:
        return None, None
    return best.r_st, best.r_en

starts = []
ends = []
for acc, sero in zip(df["Accession"], df["serotype"]):
    s, e = get_position(acc, sero)
    starts.append(s)
    ends.append(e)

df["g_start"] = starts
df["g_end"] = ends
print(df["g_start"].isna().sum())

0


In [15]:
E_START = 935
E_END = 2420

def e_overlap(row):
    if pd.isna(row["g_start"]):
        return 0
    return max(0, min(row["g_end"], E_END) - max(row["g_start"], E_START))

df["e_overlap"] = df.apply(e_overlap, axis=1)
df["span"] = df["g_end"] - df["g_start"]

In [16]:
def get_region(row):
    if row["Length"] > 9000:
        return "complete genome"
    if pd.isna(row["span"]):
        return "unmapped"
    if row["e_overlap"] / row["span"] >= 0.8:
        return "E gene"
    if row["e_overlap"] >= 1000:
        return "long, contains E"
    return "not E"

df["region"] = df.apply(get_region, axis=1)
print(df["region"].value_counts())

region
E gene              316
not E               222
complete genome     102
long, contains E     24
Name: count, dtype: int64


In [17]:
df.to_csv("/content/dengue_step2_region.csv", index=False)

In [18]:
# fraction of bases that are not A, C, G, T
def n_fraction(acc):
    seq = seqs[acc]
    bad = sum(1 for base in seq if base not in "ACGT")
    return bad / len(seq)

df["n_frac"] = df["Accession"].apply(n_fraction)
df["many_n"] = df["n_frac"] > 0.01

df["seq"] = df["Accession"].map(seqs)
df["duplicate"] = df.duplicated(subset=["seq"], keep="first")

print(df["many_n"].sum())
print(df["duplicate"].sum())

27
198


In [19]:
df["no_year"] = df["year"].isna()
print(df["no_year"].sum())
print(df[df["no_year"]][["Accession", "serotype", "Collection_Date", "Submitters"]].head(14))

14
    Accession serotype Collection_Date  \
646  DQ401689   DENV-3             NaN   
647  DQ401691   DENV-3             NaN   
648  DQ401692   DENV-3             NaN   
649  DQ401693   DENV-3             NaN   
650  AY656669   DENV-3             NaN   
651  AY656670   DENV-3             NaN   
652  AY656671   DENV-3             NaN   
653  AY656672   DENV-3             NaN   
654  AY656673   DENV-3             NaN   
655  AY656674   DENV-3             NaN   
657  AY496872   DENV-3             NaN   
660  AY496875   DENV-3             NaN   
661  AY496876   DENV-3             NaN   
663  AY496878   DENV-3             NaN   

                                            Submitters  
646  Islam,M.A., Khan,A.H., Ahmed,M.U., Begum,N., C...  
647  Islam,M.A., Khan,A.H., Ahmed,M.U., Begum,N., C...  
648  Islam,M.A., Khan,A.H., Ahmed,M.U., Begum,N., C...  
649  Islam,M.A., Khan,A.H., Ahmed,M.U., Begum,N., C...  
650  Podder,G., Breiman,R.F., Azim,T., Thu,H.M., Ve...  
651  Podder,G., Breiman,

In [20]:
df["has_E"] = df["region"].isin(["E gene", "complete genome", "long, contains E"])

# E gene sequences that are shorter than 1400 are only partial E
df["partial_E"] = (df["region"] == "E gene") & (df["Length"] < 1400)

print(df["has_E"].sum())
print(df["partial_E"].sum())

442
1


In [21]:
E_all = df[df["has_E"]]
E_good = E_all[(~E_all["many_n"]) & (~E_all["no_year"]) & (~E_all["partial_E"])]
E_unique = E_good[~E_good["duplicate"]]

print("E-containing:", len(E_all))
print("after quality filter (for counting):", len(E_good))
print("after removing duplicates (for trees):", len(E_unique))

print(pd.crosstab(E_good["year"], E_good["serotype"]))
print(pd.crosstab(E_unique["year"], E_unique["serotype"]))

E-containing: 442
after quality filter (for counting): 400
after removing duplicates (for trees): 269
serotype  DENV-1  DENV-2  DENV-3  DENV-4
year                                    
2001.0         0       0       1       0
2002.0         0       0       4       0
2004.0         0       1       0       0
2005.0         0       1       0       0
2006.0         0       2      10       0
2007.0         0       1       3       0
2008.0         0       3       4       0
2009.0         1       3       2       0
2011.0         0       1       0       0
2017.0         7      54       8       0
2018.0         5      30      24       0
2019.0         4       1      49       0
2020.0         0       0       1       0
2021.0         0       0      96       0
2022.0         0       0      39       0
2023.0         0      16       1       0
2024.0         0      11       5       1
2025.0         0       6       5       0
serotype  DENV-1  DENV-2  DENV-3  DENV-4
year                                 

In [22]:
df.drop(columns=["seq"]).to_csv("/content/dengue_step3_flags.csv", index=False)

In [23]:
print(df[df["no_year"]][["Accession", "Isolate", "Release_Date", "Length"]])

    Accession          Isolate Release_Date  Length
646  DQ401689  mutant BDH02-01   2006-03-15   10707
647  DQ401691  mutant BDH02-03   2006-03-15   10707
648  DQ401692  mutant BDH02-04   2006-03-15   10707
649  DQ401693  mutant BDH02-07   2006-03-15   10707
650  AY656669              114   2005-11-26    1479
651  AY656670              164   2005-11-26    1479
652  AY656671              165   2005-11-26    1479
653  AY656672              Apu   2005-11-26    1479
654  AY656673            Jacob   2005-11-26    1479
655  AY656674              058   2005-11-26    1479
657  AY496872          BDH02-2   2005-03-05    3033
660  AY496875          BDH02-5   2005-03-05    3033
661  AY496876          BDH02-6   2005-03-05    3033
663  AY496878          BDH02-8   2005-03-05    3033


In [24]:
print(pd.crosstab(df["year"], df["many_n"]))
print(df[df["many_n"]][["Accession", "year", "serotype", "Length", "n_frac"]].sort_values("year").head(15))

many_n  False  True 
year                
2001.0      1      0
2002.0      4      0
2004.0      1      0
2005.0      1      0
2006.0     12      0
2007.0      4      0
2008.0      7      0
2009.0      6      0
2011.0      1      0
2017.0    136      0
2018.0     90      0
2019.0     70      0
2020.0      1      0
2021.0    101      0
2022.0     78      0
2023.0     79      0
2024.0     20      0
2025.0     11     27
   Accession    year serotype  Length    n_frac
0   OZ484492  2025.0   DENV-2   10512  0.024068
1   OZ484493  2025.0   DENV-3   10461  0.140618
2   OZ484494  2025.0   DENV-2   10512  0.152778
3   OZ484495  2025.0   DENV-2   10512  0.063927
4   OZ484496  2025.0   DENV-2   10512  0.041096
5   OZ484497  2025.0   DENV-2   10512  0.115963
6   OZ484498  2025.0   DENV-2   10512  0.099981
7   OZ484499  2025.0   DENV-2   10512  0.082097
8   OZ484500  2025.0   DENV-3   10461  0.091100
9   OZ484501  2025.0   DENV-1   10477  0.136585
10  OZ484502  2025.0   DENV-1   10477  0.050014
11  

In [25]:
E_all = df[df["has_E"]]

E_count = E_all[(~E_all["many_n"]) & (~E_all["no_year"]) & (~E_all["partial_E"])]
E_tree = E_all[(~E_all["many_n"]) & (~E_all["partial_E"]) & (~E_all["duplicate"])]

print("for counting:", len(E_count))
print("for trees:", len(E_tree))
print(E_tree["serotype"].value_counts())

for counting: 400
for trees: 282
serotype
DENV-3    163
DENV-2    103
DENV-1     15
DENV-4      1
Name: count, dtype: int64


In [26]:
df["year_note"] = ""
mask = df["Isolate"].astype(str).str.contains("BDH02") & df["year"].isna()
df.loc[mask, "year"] = 2002
df.loc[mask, "year_note"] = "from isolate name, to verify"

print(mask.sum())
print(df["year"].isna().sum())

8
6


In [27]:
e_ref = {}
for sero in ["DENV-1", "DENV-2", "DENV-3"]:
    pool = df[(df["serotype"] == sero) & (df["region"] == "E gene") & (df["Length"].isin([1479, 1485]))]
    best = pool.sort_values("n_frac").iloc[0]["Accession"]
    e_ref[sero] = best
    with open(f"/content/Eref_{sero}.fa", "w") as f:
        f.write(f">{best}\n{seqs[best]}\n")

print(e_ref)

{'DENV-1': 'OQ826841', 'DENV-2': 'PZ017579', 'DENV-3': 'PZ017435'}


In [28]:
e_aligners = {}
for sero in e_ref:
    e_aligners[sero] = mp.Aligner(f"/content/Eref_{sero}.fa", preset="map-ont")

e_seq = {}
for acc, sero, has_e in zip(df["Accession"], df["serotype"], df["has_E"]):
    if (not has_e) or (sero not in e_aligners):
        continue
    hits = list(e_aligners[sero].map(seqs[acc]))
    if len(hits) == 0:
        continue
    best = max(hits, key=lambda h: h.mlen)
    e_seq[acc] = seqs[acc][best.q_st:best.q_en]

df["E_seq"] = df["Accession"].map(e_seq)
df["E_len"] = df["E_seq"].str.len()

print(df["E_seq"].notna().sum())
print(df["E_len"].value_counts().head(8))

439
E_len
1479.0    262
1485.0    144
1484.0     19
1478.0      7
1442.0      2
1406.0      2
847.0       1
1443.0      1
Name: count, dtype: int64


In [29]:
def n_fraction_E(s):
    if not isinstance(s, str):
        return None
    return sum(1 for b in s if b not in "ACGT") / len(s)

df["E_n_frac"] = df["E_seq"].apply(n_fraction_E)
df["E_ok"] = (df["E_len"] >= 1400) & (df["E_n_frac"] <= 0.01)

print(df["E_ok"].sum())
print(df[df["has_E"] & ~df["E_ok"]][["Accession", "serotype", "region", "Length", "E_len", "E_n_frac"]])

430
    Accession serotype           region  Length   E_len  E_n_frac
1    OZ484493   DENV-3  complete genome   10461  1479.0  0.133198
8    OZ484500   DENV-3  complete genome   10461  1479.0  0.133198
9    OZ484501   DENV-1  complete genome   10477  1485.0  0.158249
11   OZ484503   DENV-4  complete genome   10511     NaN       NaN
12   OZ484504   DENV-4  complete genome   10511     NaN       NaN
14   OZ484506   DENV-1  complete genome   10477  1484.0  0.158356
21   OZ484513   DENV-3  complete genome   10461   847.0  0.232586
23   OZ484515   DENV-3  complete genome   10461  1479.0  0.150101
24   OZ484516   DENV-3  complete genome   10461  1479.0  0.133198
25   OZ484517   DENV-3  complete genome   10461  1479.0  0.302231
37   PZ017578   DENV-4           E gene    1584     NaN       NaN
117  PP697469   DENV-2           E gene     684   568.0  0.000000


In [30]:
E_count = df[df["E_ok"] & df["year"].notna()]
E_tree = df[df["E_ok"]].drop_duplicates(subset="E_seq")

print(len(E_count), len(E_tree))
print(E_tree["serotype"].value_counts())
print(pd.crosstab(E_count["year"], E_count["serotype"]))

424 252
serotype
DENV-3    136
DENV-2     99
DENV-1     17
Name: count, dtype: int64
serotype  DENV-1  DENV-2  DENV-3
year                            
2001.0         0       0       1
2002.0         0       0      12
2004.0         0       1       0
2005.0         0       1       0
2006.0         0       2      10
2007.0         0       1       3
2008.0         0       3       4
2009.0         1       3       2
2011.0         0       1       0
2017.0         7      54       8
2018.0         5      30      24
2019.0         4       1      49
2020.0         0       0       1
2021.0         0       0      96
2022.0         0       0      39
2023.0         0      16       1
2024.0         0      11       5
2025.0         3      20       5


In [31]:
with open("/content/E_tree.fasta", "w") as f:
    for _, r in E_tree.iterrows():
        f.write(f">{r['Accession']}|{r['serotype']}|{r['year']}\n{r['E_seq']}\n")

df.drop(columns=["seq", "E_seq"]).to_csv("/content/dengue_step4_E.csv", index=False)
E_tree.drop(columns=["seq", "E_seq"]).to_csv("/content/E_tree_info.csv", index=False)

In [32]:
d2 = df[df["serotype"] == "DENV-2"]
print(pd.crosstab(d2["year"], d2["region"]))

not_e = df[df["region"] == "not E"]
print(pd.crosstab(not_e["year"], not_e["Length"]))

region  E gene  complete genome  long, contains E  not E
year                                                    
2004.0       1                0                 0      0
2005.0       1                0                 0      0
2006.0       2                0                 0      0
2007.0       1                0                 0      0
2008.0       3                0                 0      0
2009.0       3                0                 0      0
2011.0       1                0                 0      0
2017.0      47                7                 0     51
2018.0      30                0                 0     19
2019.0       0                1                 0      1
2022.0       0                0                 0      2
2023.0       3                1                13     57
2024.0      10                1                 0      3
2025.0       6               14                 0      0
Length  150   151   208   232   260   341   352   353   372   383   ...  449   \
year   

In [33]:
!pip install zstandard -q

import io
import urllib.request
import zstandard

def load_metadata(url):
    data = urllib.request.urlopen(url).read()
    text = zstandard.ZstdDecompressor().decompressobj().decompress(data)
    return pd.read_csv(io.BytesIO(text), sep="\t", low_memory=False)

meta = {}
for s in ["denv1", "denv2", "denv3"]:
    url = f"https://data.nextstrain.org/files/workflows/dengue/metadata_{s}.tsv.zst"
    meta[s] = load_metadata(url)
    print(s, meta[s].shape)

print(meta["denv3"].columns.tolist())
print(meta["denv3"].head(3).T)

denv1 (21928, 39)
denv2 (20303, 39)
denv3 (10008, 39)
['accession', 'strain', 'accession_version', 'date', 'region', 'country', 'division', 'location', 'length', 'host', 'host_genus', 'host_type', 'is_lab_host', 'date_released', 'date_updated', 'authors', 'full_authors', 'institution', 'serotype_genbank', 'url', 'genotype_nextclade', 'alignmentStart', 'alignmentEnd', 'genome_coverage', 'failedCdses', 'E_coverage', 'C_coverage', 'prM_coverage', 'NS1_coverage', 'NS2A_coverage', 'NS2B_coverage', 'NS3_coverage', 'NS4A_coverage', '2K_coverage', 'NS4B_coverage', 'NS5_coverage', 'genotype', 'major_lineage', 'minor_lineage']
                                                                    0  \
accession                                                    PV554919   
strain                                             AIIMSR-DEN-49/2024   
accession_version                                          PV554919.1   
date                                                       2024-08-30   
region    

In [34]:
for s in meta:
    m = meta[s]
    found = m.astype(str).apply(lambda col: col.str.contains("Bangladesh", case=False)).any(axis=1)
    print(s, found.sum())

denv1 33
denv2 278
denv3 355


In [35]:
short = df[df["Length"] <= 160]
print(short[["Accession", "serotype", "year", "Isolate", "Submitters"]].head(8))
print(short["Submitters"].str[:40].value_counts())

   Accession serotype    year     Isolate  \
55  PQ621312   DENV-2  2023.0  S20/DENV-2   
56  PQ621313   DENV-2  2023.0  S18/DENV-2   
57  PQ621314   DENV-2  2023.0  S16/DENV-2   
58  PQ621315   DENV-2  2023.0  S15/DENV-2   
59  PQ621316   DENV-2  2023.0  S14/DENV-2   
60  PQ621317   DENV-2  2023.0  S13/DENV-2   
61  PQ621318   DENV-2  2023.0  S12/DENV-2   
62  PQ621319   DENV-2  2023.0  S11/DENV-2   

                                           Submitters  
55  Akter,L., Rakib,T.M., Rana,E.A., Alam,M., Khal...  
56  Akter,L., Rakib,T.M., Rana,E.A., Alam,M., Khal...  
57  Akter,L., Rakib,T.M., Rana,E.A., Alam,M., Khal...  
58  Akter,L., Rakib,T.M., Rana,E.A., Alam,M., Khal...  
59  Akter,L., Rakib,T.M., Rana,E.A., Alam,M., Khal...  
60  Akter,L., Rakib,T.M., Rana,E.A., Alam,M., Khal...  
61  Akter,L., Rakib,T.M., Rana,E.A., Alam,M., Khal...  
62  Akter,L., Rakib,T.M., Rana,E.A., Alam,M., Khal...  
Submitters
Akter,L., Rakib,T.M., Rana,E.A., Alam,M.    42
Name: count, dtype: int64


In [36]:
all_meta = pd.concat([
    meta["denv1"].assign(file="denv1"),
    meta["denv2"].assign(file="denv2"),
    meta["denv3"].assign(file="denv3"),
])

wrong_acc = df[df["label_wrong"]]["Accession"]
x = all_meta[all_meta["accession"].isin(wrong_acc)]

print(len(x))
print(x["file"].value_counts())
print(x["serotype_genbank"].value_counts())
print(x["genotype"].value_counts(dropna=False))

44
file
denv3    44
Name: count, dtype: int64
serotype_genbank
denv3    44
Name: count, dtype: int64
genotype
3I    44
Name: count, dtype: int64


In [37]:
ns = all_meta[["accession", "file", "genotype", "major_lineage", "minor_lineage", "E_coverage"]]
ns = ns.drop_duplicates("accession")
ns = ns.rename(columns={
    "accession": "Accession",
    "file": "ns_file",
    "genotype": "ns_genotype",
    "major_lineage": "ns_major",
    "minor_lineage": "ns_minor",
})

df = df.merge(ns, on="Accession", how="left")

print(df["ns_genotype"].notna().sum())
print(pd.crosstab(df["serotype"], df["ns_genotype"].fillna("none")))

661
ns_genotype  1I  1III  2II  2V   3I  3II  3III  none
serotype                                            
DENV-1        5    24    0   0    0    0     0     0
DENV-2        0     0  238  40    0    0     0     0
DENV-3        0     0    0   0  297   38    19     0
DENV-4        0     0    0   0    0    0     0     3


In [38]:
df["len_group"] = pd.cut(df["Length"], [0, 160, 600, 1200, 1600, 20000],
                         labels=["150", "160-600", "600-1200", "1200-1600", ">1600"])

print(pd.crosstab([df["serotype"], df["ns_genotype"]], df["len_group"]))

len_group             150  160-600  600-1200  1200-1600  >1600
serotype ns_genotype                                          
DENV-1   1I             0        0         0          0      5
         1III           0        0         7         15      2
DENV-2   2II            3       40        52         95     48
         2V            39        0         0          0      1
DENV-3   3I             0       74         7        166     50
         3II            0        0         0         26     12
         3III           0        1         0          0     18


In [39]:
good = df[df["Length"] >= 400]

for sero in ["DENV-1", "DENV-2", "DENV-3"]:
    sub = good[good["serotype"] == sero]
    print(sero)
    print(pd.crosstab(sub["year"], sub["ns_genotype"]))
    print()

DENV-1
ns_genotype  1I  1III
year                 
2009.0        0     1
2017.0        0    14
2018.0        0     5
2019.0        0     4
2025.0        5     0

DENV-2
ns_genotype  2II  2V
year                
2004.0         1   0
2005.0         1   0
2006.0         2   0
2007.0         1   0
2008.0         3   0
2009.0         3   0
2011.0         1   0
2017.0       105   0
2018.0        43   0
2019.0         2   0
2022.0         2   0
2023.0        30   1
2024.0        14   0
2025.0        20   0

DENV-3
ns_genotype  3I  3II  3III
year                      
2001.0        0    1     0
2002.0        0   12     0
2006.0        0   10     0
2007.0        0    3     0
2008.0        0    4     0
2009.0        0    2     0
2017.0       15    0     0
2018.0       35    0     0
2019.0       63    0     1
2020.0        1    0     0
2021.0       82    0    18
2022.0       72    0     0
2023.0        1    0     0
2024.0        5    0     0
2025.0       11    0     0



In [40]:
print(df[(df["ns_genotype"] == "2V") & (df["Length"] > 160)][["Accession", "year", "Length", "n_frac", "Submitters"]])

g3 = df[df["ns_genotype"] == "3III"]
print(g3[["Accession", "year", "Length", "Isolate"]].head(20))
print(g3["Submitters"].str[:40].value_counts())

   Accession    year  Length  n_frac  \
98  PQ657766  2023.0   10723     0.0   

                                           Submitters  
98  Mannan,A., Rob,M.A., Satter,M., Faisal,A., Cho...  
    Accession    year  Length   Isolate
378  ON103302  2021.0   10707       NaN
380  ON103390  2021.0   10707       NaN
382  ON111445  2021.0   10707       NaN
384  ON115021  2021.0   10707       NaN
386  ON115023  2021.0   10707       NaN
388  ON115025  2021.0   10707       NaN
390  ON115029  2021.0   10707       NaN
392  ON115182  2021.0   10707       NaN
394  ON115811  2021.0   10707       NaN
396  ON115814  2021.0   10707       NaN
398  ON115816  2021.0   10707       NaN
400  ON115818  2021.0   10707       NaN
402  ON116129  2021.0   10707       NaN
404  ON127418  2021.0   10707       NaN
406  ON127533  2021.0   10707       NaN
408  ON127537  2021.0   10707       NaN
410  ON127550  2021.0   10707       NaN
412  ON127552  2021.0   10707       NaN
443  MW599417  2019.0     444  Khulna02
Submitt

In [41]:
df["geno_ok"] = df["Length"] >= 400
print(df["geno_ok"].sum())

603


In [42]:
good = df[df["geno_ok"]]

for sero in ["DENV-2", "DENV-3"]:
    sub = good[good["serotype"] == sero]
    print(sero)
    print(pd.crosstab(sub["ns_minor"], sub["year"]))
    print()

DENV-2
year         2004.0  2005.0  2006.0  2007.0  2008.0  2009.0  2011.0  2017.0  \
ns_minor                                                                      
2II_A.1           1       1       2       1       3       3       1       0   
2II_A.2.1         0       0       0       0       0       0       0       8   
2II_F.1.1         0       0       0       0       0       0       0       0   
2II_F.1.1.2       0       0       0       0       0       0       0      97   
2II_F.1.2         0       0       0       0       0       0       0       0   
2V_F              0       0       0       0       0       0       0       0   

year         2018.0  2019.0  2022.0  2023.0  2024.0  2025.0  
ns_minor                                                     
2II_A.1           0       0       0       0       0       0  
2II_A.2.1         0       0       0       0       0       0  
2II_F.1.1         0       0       0      30      14      20  
2II_F.1.1.2      43       2       0       0       

In [43]:
df.drop(columns=["seq", "E_seq"]).to_csv("/content/dengue_step5_genotype.csv", index=False)